# Workflow Test: Vacuum Gripper
### Geometries, Gripper, Grasp Sampler

In [12]:
import os
from os.path import join as pjoin
import sys
import numpy as np
import open3d as o3d
import yaml
from copy import deepcopy

# Import our custom modules
from src.grippers.vacuum_gripper import VacuumGripper, VacuumGripperConfig
from src.grasping.vacuum_sampler import VacuumGraspSampler, VacuumSamplerConfig
import src.utils.geometry_utils as gu
from src.generic_geometry import GenericGeometry


### Load Configs and data

In [13]:
franka_path = pjoin("gripper_parameter", "franka_vacuum.yaml")
object_path = pjoin("Test_part", "pringles", "clouds","merged_cloud.ply")

gripper = VacuumGripper(franka_path)
pcd = GenericGeometry(object_path)


✅ Geometry set: mesh (open3d)
🔧 Vacuum Gripper 'Schmalz_ECG_Vacuum' initialized.
   - Body Dimensions: R=75.8mm, L=88.6mm
[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
[Open3D] Loaded PointCloud: Test_part/pringles/clouds/merged_cloud.ply
✅ Geometry set: point_cloud (open3d)


In [14]:
gripper.visualize()
pcd.visualize()

## Create Grasp Sampler

In [15]:
# 2. Initialize Sampler Configuration
sampler_config = VacuumSamplerConfig(
    num_samples=800,        # Cast 300 rays
    approach_distance=0.05, # 5cm approach check
    max_curvature=0.08,     # Tolerance for curvature
    min_score=0.4           # Minimum GSS to accept
)

# 3. Initialize Sampler
sampler = VacuumGraspSampler(gripper, sampler_config)

### Sample Grasps

In [16]:
pcd.get_dimensions_report()

{'extents_xyz': array([0.0868, 0.0857, 0.2444]),
 'diagonal': 0.2731,
 'likely_unit': 'meters',
 'suggested_voxel_size': 0.00273,
 'details': 'Type: Open3D PointCloud. Size: 0.09 x 0.09 x 0.24. Unit: METERS.'}

In [17]:
pcd_down = pcd.downsample(voxel_size=0.003)
GenericGeometry(geometry=pcd_down).visualize()

Downsampling with voxel_size=0.003...
✅ Geometry set: point_cloud (open3d)


In [7]:
sampler.config

VacuumSamplerConfig(num_samples=800, approach_distance=0.05, max_curvature=0.08, max_angle_deg=45.0, min_score=0.4, weight_flatness=0.4, weight_verticality=0.3, weight_torque=0.3)

In [8]:
# sampler.config.weight_verticality = 5
# sampler.config.weight_flatness = 0
# sampler.config.weight_torque = 0

In [18]:
# Run the sampling pipeline
pcd_down = pcd.downsample(voxel_size=0.003)
grasps = sampler.sample_grasps(pcd_down)

print(f"\nResult: Found {len(grasps)} valid grasps.")

if len(grasps) > 0:
    best_grasp = grasps[0]
    print(f"Top Grasp Score: {best_grasp.score:.4f}")
    print(f"Top Grasp Position: {best_grasp.contact_point}")
else:
    print("No valid grasps found. Check thresholds.")

Downsampling with voxel_size=0.003...
[VacuumSampler] Phase 1: Found 800 collision-free TCPs.
[VacuumSampler] Result: 9 valid grasps (from 800 raw candidates).

Result: Found 9 valid grasps.
Top Grasp Score: 0.7456
Top Grasp Position: [-0.01341788  0.01428263  0.24416607]


In [19]:
for c in sampler.get_best_candidates(9):
    sampler.visualize_grasp(pcd_down, c)

✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)
✅ Geometry set: mesh (open3d)


In [6]:
sampler.get_best_candidates(1)

[GraspCandidate(transform=array([[ 0.99898152, -0.02808203, -0.0353174 , -0.01341788],
        [ 0.02709645,  0.9992383 , -0.02808203,  0.01428263],
        [ 0.0360791 ,  0.02709645,  0.99898152,  0.24416607],
        [ 0.        ,  0.        ,  0.        ,  1.        ]]), score=np.float64(0.7456106596436125), contact_point=array([-0.01341788,  0.01428263,  0.24416607]), approach_vector=array([-0.0353174 , -0.02808203,  0.99898152]), score_details={'flatness': np.float64(0.5326796179841802), 'verticality': np.float64(0.9425304433024383), 'torque': np.float64(0.8325989315306968), 'raw_angle_deg': np.float64(2.586130051390277)})]

In [7]:
sampler.valid_candidates[0]

GraspCandidate(transform=array([[ 0.99898152, -0.02808203, -0.0353174 , -0.01341788],
       [ 0.02709645,  0.9992383 , -0.02808203,  0.01428263],
       [ 0.0360791 ,  0.02709645,  0.99898152,  0.24416607],
       [ 0.        ,  0.        ,  0.        ,  1.        ]]), score=np.float64(0.7456106596436125), contact_point=array([-0.01341788,  0.01428263,  0.24416607]), approach_vector=array([-0.0353174 , -0.02808203,  0.99898152]), score_details={'flatness': np.float64(0.5326796179841802), 'verticality': np.float64(0.9425304433024383), 'torque': np.float64(0.8325989315306968), 'raw_angle_deg': np.float64(2.586130051390277)})

In [24]:
sampler.visualize_candidates_heatmap(pcd_down, attribute='torque', valid_only=False)